# 05 — CNN profunda + balanceo + umbral óptimo

Último peldaño de la escalera: mismo notebook que 04, añadiendo la optimización del umbral de decisión por clase (en vez de fijarlo en 0.5). 

## 1. Entorno, semillas y librerías

In [1]:
import sys, os, random, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal
from scipy.signal import sosfiltfilt, iirnotch, tf2sos

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.regularizers import l2

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general


In [2]:
NOTEBOOK_NAME = "05_cnn_profunda_balanceo_umbral"
PREP_VERSION  = "con_filtrado"     # reutiliza el mismo caché que 02, 03 y 04, la señal no cambia

DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / PREP_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 40.0
NOTCH_FREQ    = 50.0
NOTCH_Q       = 30.0

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)

L2_REG = 1e-4   # fuerza de la regularización L2
GAMMA  = 1.0    # factor de enfoque de la focal loss


## 3. Carga de metadatos y etiquetas multi-etiqueta

In [3]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)


## 4. Split por paciente

In [4]:
train_df = df[df["strat_fold"].isin(range(1, 9))].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

#Para comprobar que ningún paciente está en dos conjuntos 
assert set(train_df.patient_id) & set(val_df.patient_id) == set()
assert set(train_df.patient_id) & set(test_df.patient_id) == set()
assert set(val_df.patient_id)  & set(test_df.patient_id)  == set()

print("Tamaños:", len(train_df), len(val_df), len(test_df))


Tamaños: 17084 2146 2158


## 5. Preprocesado de la señal ECG (limpieza)

Idéntico al `preprocess_ecg` del notebook principal: filtro paso-banda de fase cero (0.5-40Hz), notch a 50Hz, y normalización z-score por derivación. Es la única función nueva respecto a 01.

In [5]:
def is_valid_signal(x, flat_std_thresh=1e-6):
    """Descarta derivaciones corruptas: con NaN/Inf o completamente planas (fallo de electrodo)."""
    if np.any(np.isnan(x)) or np.any(np.isinf(x)):
        return False
    return np.std(x) >= flat_std_thresh

_BP_SOS    = sp_signal.butter(4, [BANDPASS_LOW / (FS/2), BANDPASS_HIGH / (FS/2)], btype="band", output="sos")
_NOTCH_SOS = tf2sos(*iirnotch(NOTCH_FREQ / (FS/2), NOTCH_Q))

def clean_lead(x):
    """Limpia una derivación (1D): paso-banda + notch + normalización z-score."""
    x = np.asarray(x, dtype=np.float64)
    if not is_valid_signal(x):
        return None
    x = sosfiltfilt(_BP_SOS, x)
    x = sosfiltfilt(_NOTCH_SOS, x)
    mu, sigma = x.mean(), x.std()
    if sigma < 1e-8:
        return None
    return ((x - mu) / sigma).astype(np.float32)

def preprocess_ecg(raw_signal):
    """Limpia las 12 derivaciones de un ECG. Descarta el registro si más de la mitad fallan."""
    cleaned = np.zeros_like(raw_signal, dtype=np.float32)
    n_bad = 0
    for i in range(raw_signal.shape[1]):
        c = clean_lead(raw_signal[:, i])
        cleaned[:, i] = c if c is not None else 0.0
        n_bad += (c is None)
    return None if n_bad > raw_signal.shape[1] // 2 else cleaned


## 6. Extracción de características: espectrograma STFT


In [8]:
def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    """Convierte una derivación 1D en su espectrograma STFT normalizado, con forma fija (F, T)."""
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))

    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad

    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    """signal_2d: (n_muestras, 12) -> tensor (F, T, 12)."""
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)


## 7. Generación del dataset con caché en disco


In [9]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    clean = preprocess_ecg(raw)
    if clean is None:
        return None
    tensor = ecg_to_tensor(clean)
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        if t is None:
            continue
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_train, Y_train = build_split(train_df)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)

print("Formas:", X_train.shape, X_val.shape, X_test.shape)


Formas: (17084, 33, 64, 12) (2146, 33, 64, 12) (2158, 33, 64, 12)


## 8. Arquitectura: CNN de tres bloques


In [10]:
def build_model_deep(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=l2(L2_REG))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)

    return models.Model(inputs, outputs)

model = build_model_deep(X_train.shape[1:], len(SUPER_CLASSES))
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 33, 64, 12)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 33, 64, 32)          │           3,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 33, 64, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 16, 32, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 16, 32, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 8, 16, 64)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 8, 16, 128)          │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 8, 16, 128)          │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 105,317 (411.39 KB)

 Trainable params: 104,869 (409.64 KB)

 Non-trainable params: 448 (1.75 KB)

## 9. Balanceo de clases (pos_weight + focal loss)


In [11]:
pos_counts = Y_train.sum(axis=0)
neg_counts = len(Y_train) - pos_counts
pos_weight = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
print("pos_weight por clase:", dict(zip(SUPER_CLASSES, pos_weight.round(2))))

pos_weight_tf = tf.constant(pos_weight, dtype=tf.float32)

def focal_weighted_bce(y_true, y_pred, gamma=GAMMA):
    """Entropía cruzada binaria ponderada por clase (pos_weight) y modulada por dificultad (focal, gamma)."""
    eps = 1e-7
    y_pred = tf.clip_by_value(y_pred, eps, 1 - eps)
    pos_term = pos_weight_tf * y_true * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
    neg_term = (1 - y_true) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
    return tf.reduce_mean(-(pos_term + neg_term))

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal_weighted_bce,
              metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])


pos_weight por clase: {'NORM': 1.25, 'MI': 2.9, 'STTC': 3.08, 'HYP': 7.06, 'CD': 3.37}


Un error en una hipertrofia pesa 7 veces más que en normal 

## 10. Entrenamiento


In [12]:
cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(str(MODEL_DIR / f"{NOTEBOOK_NAME}.keras"), monitor="val_auc",
                               mode="max", save_best_only=True),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=60,
    batch_size=32,
    callbacks=cbs,
    verbose=1,
)


Epoch 1/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - auc: 0.7802 - loss: 0.4435 - val_auc: 0.8450 - val_loss: 0.3908 - learning_rate: 0.0010
Epoch 2/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - auc: 0.8413 - loss: 0.3910 - val_auc: 0.8551 - val_loss: 0.3897 - learning_rate: 0.0010
Epoch 3/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 38s 70ms/step - auc: 0.8555 - loss: 0.3749 - val_auc: 0.8621 - val_loss: 0.3763 - learning_rate: 0.0010
Epoch 4/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - auc: 0.8623 - loss: 0.3674 - val_auc: 0.8694 - val_loss: 0.3648 - learning_rate: 0.0010
Epoch 5/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 42s 74ms/step - auc: 0.8695 - loss: 0.3586 - val_auc: 0.8730 - val_loss: 0.3563 - learning_rate: 0.0010
Epoch 6/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - auc: 0.8742 - loss: 0.3519 - val_auc: 0.8749 - val_loss: 0.3561 - learning_rate: 0.0010
Epoch 7/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 41s 74ms/step - auc: 0.8770 - loss: 0.3489 - val_auc: 0.8712 - val_loss: 0.3933 - learning_rate:

## 11. Optimización del umbral de decisión por clase

Para cada clase se recorre la curva precisión-recall sobre el conjunto de **validación** y se elige el umbral que maximiza el F1. Se usa validación (no test) para elegir el umbral evita que la evaluación final esté sesgada por la propia optimización.

In [13]:
val_probs = model.predict(X_val, verbose=0)

best_thresholds = {}
for i, c in enumerate(SUPER_CLASSES):
    prec, rec, thr = precision_recall_curve(Y_val[:, i], val_probs[:, i])
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-8, None)
    best_i = np.nanargmax(f1[:-1]) if len(thr) > 0 else 0
    best_thresholds[c] = float(thr[best_i]) if len(thr) > 0 else 0.5

print("Umbrales óptimos por clase (validación):", best_thresholds)
THRESH_VEC = np.array([best_thresholds[c] for c in SUPER_CLASSES])


Umbrales óptimos por clase (validación): {'NORM': 0.4524393677711487, 'MI': 0.42697495222091675, 'STTC': 0.6164358854293823, 'HYP': 0.6522840261459351, 'CD': 0.7015767097473145}


## 12. Evaluación sobre el conjunto de test

In [14]:
def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

test_probs = model.predict(X_test, verbose=0)
test_pred  = (test_probs >= THRESH_VEC).astype(int)

metrics_df = compute_metrics(Y_test, test_pred, test_probs)
metrics_df


,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.831789,0.755973,0.920042,0.829977,0.266208
1,Infarto,0.783596,0.553966,0.774545,0.645944,0.332317
2,Isquemia/ST-T,0.854495,0.661972,0.811900,0.729310,0.295048
3,Hipertrofia,0.869787,0.465704,0.492366,0.478664,0.385338
4,Conducción (bloqueos),0.864690,0.751232,0.614919,0.676275,0.333178
5,Macro,0.840871,0.637769,0.722755,0.672034,0.322418


## 13. Registro en la tabla comparativa común


In [15]:
macro_row = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0]
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION,
    "Parametros_entrenables": model.count_params(),
    **macro_row.to_dict(),
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV)
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row

comparison.to_csv(COMPARISON_CSV, index=False)
comparison


,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
1,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
2,04_cnn_profunda_balanceo,con_filtrado,105317,0.810936,0.577661,0.792424,0.656731,0.286431
3,05_cnn_profunda_balanceo_umbral,con_filtrado,105317,0.840871,0.637769,0.722755,0.672034,0.322418


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path

nombres = [CLASS_MAP_HUMAN[c] for c in SUPER_CLASSES]
n = len(SUPER_CLASSES)

una_clase = Y_test.sum(axis=1) == 1
y_real = Y_test[una_clase].argmax(axis=1)
y_pred = (test_probs[una_clase] / np.asarray(THRESH_VEC)).argmax(axis=1)

cm = np.zeros((n, n), dtype=int)
for r, p in zip(y_real, y_pred):
    cm[r, p] += 1
cm_pct = cm / cm.sum(axis=1, keepdims=True)

print(f"ECG de prueba con una sola superclase: {una_clase.sum()} de {len(Y_test)}")
print(f"Aciertos totales: {np.trace(cm)} de {cm.sum()} ({np.trace(cm) / cm.sum():.1%})")
for i in range(n):
    fila = cm_pct[i].copy()
    fila[i] = -1
    j = fila.argmax()
    print(f"{nombres[i]:<24} n = {cm[i].sum():4d}   acierto {cm_pct[i, i]:.1%}   principal confusión: {nombres[j]} ({cm_pct[i, j]:.1%})")
print("\nMatriz de recuentos (filas = real, columnas = predicha):")
print(cm)

fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
for i in range(n):
    for j in range(n):
        texto = f"{cm_pct[i, j]:.1%}".replace(".", ",") + f"\n({cm[i, j]})"
        ax.text(j, i, texto, ha="center", va="center", fontsize=9,
                color="white" if cm_pct[i, j] > 0.55 else "#1f2a33")
ax.set_xticks(range(n))
ax.set_xticklabels(nombres, rotation=30, ha="right")
ax.set_yticks(range(n))
ax.set_yticklabels(nombres)
ax.set_xlabel("Superclase predicha")
ax.set_ylabel("Superclase real")
barra = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
barra.ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
plt.tight_layout()
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("outputs/figures/matriz_confusion_5x5.png", dpi=220, bbox_inches="tight")
plt.show()


NameError: name 'Y_test' is not defined